In [13]:
import numpy as np
import matplotlib.pyplot as plt

### Detector class

Добавляю метод генерации фотоэлектронов в класс `Detector` по двум причинам:

- Физическая: генерация фотоэлектронов является частью отклика фотокатода на попавший в него фотон, поэтому логично отнести этот процесс к детектору.
- Программная: для генерации требуется параметр $a$, который уже является полем класса `Detector`. Поэтому метод может использовать его напрямую, без передачи параметров или обращения к объекту другого класса.

In [14]:
class Detector():
    def __init__(self, R: float, N: int, r: float, a: float):
        """
        Parameters
        ----------
        R : float
            Радиус детектора в метрах.
        N : int
            Количество фотоумножителей.
        r : float
            Радиус фотокатода в метрах.
        a : float
            Эффективность фотокатода.
        """
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.rng = np.random.default_rng()

    def coord_cart(self, theta: float, phi: float) -> np.array:
        return np.array([self.R * np.sin(theta)*np.cos(phi), self.R * np.sin(theta)*np.sin(phi), self.R*np.cos(theta)])

    def fill_detector(self):
        self.photo_multipliers = np.zeros((self.N, 3))
        S = 4 * self.R**2 * np.pi 
        delta_S = S / self.N 
        lin_size = delta_S**0.5 
        N_theta = np.pi * R / lin_size 
        delta_theta = np.pi / N_theta

        theta_floor = [int(np.floor(N_theta)), np.pi / np.floor(N_theta), delta_theta - np.pi / np.floor(N_theta)]
        theta_ceil = [int(np.ceil(N_theta)), np.pi / np.ceil(N_theta), delta_theta - np.pi / np.ceil(N_theta)]

        best_theta = min(theta_floor, theta_ceil, key=lambda x: abs(x[2]))
        N_theta, delta_theta = best_theta[0], best_theta[1]

        mid_thetas = np.array([delta_theta / 2 + delta_theta*i for i in range(0, N_theta)])
        n_phi = np.array([(np.pi * self.N)**0.5*np.sin(theta) for theta in mid_thetas])

        A = self.N / sum([np.sin(theta) for theta in mid_thetas])
        #n_phi = np.rint(np.array([A * np.sin(theta) for theta in mid_thetas])) #-> (3, 7, 10, 10, 7, 3)
        n_phi = np.array([2, 7, 11, 11, 7, 2])     

        phi_phases = np.zeros(N_theta)
        for i in range(N_theta):
            if i == 0:
                continue
            phi_phases[i] = np.pi / np.lcm(int(n_phi[i-1]), int(n_phi[i]))

        n = 0
        phase = 0
        for i in range(N_theta):
            if i < 3:
                phase += phi_phases[i]
            else:
                phase -= phi_phases[i]
            for j in range(int(n_phi[i])):
                self.photo_multipliers[n] = self.coord_cart(mid_thetas[i], 2*np.pi/n_phi[i] * j + phase)
                n += 1

    def in_chord(self, coord: np.array) -> int:
        for k, photo_multiplier in enumerate(self.photo_multipliers):
            distance = np.linalg.norm(photo_multiplier-coord)
            if distance <= self.r:
                return k
        return -1

    def photo_electrons(self) -> int:
        return self.rng.poisson(lam=self.a)

    def register(self, coord: np.array) -> int:
        id = self.in_chord(coord)
        if id < 0:
            return 0
        return self.photo_electrons()
        

### Source class

In [15]:
class Source:
    def __init__(self, pos: np.array):
        """
        Parameters
        ----------
        pos : np.array
            координаты источника
        """
        self.Y = 10000 
        self.pos = pos

    def sphere(self, R: float) -> np.array: #взял метод из ROOT
        r2 = 1.0

        while r2 > 0.25:
            a = np.random.random() - 0.5
            b = np.random.random() - 0.5
            r2 = a**2 + b**2

        z = R * (-1.0 + 8.0 * r2)
        scale = 8.0 * R * np.sqrt(0.25 - r2)

        x = a * scale
        y = b * scale

        point = np.array([x, y, z])

        return point

    def generate_photons(self, E: float) -> np.array:
        photon_number = int(E * self.Y)
        photons = np.zeros((photon_number, 3))
        for i in range(photon_number):
            photons[i] = self.sphere(1.0)
        return photons


### Задача 3
Положение фотона:
$$
\vec{r}(t) = \vec{r}_0 + t\cdot \vec{n}
$$
Условие пересечения детектора:
$$
|\vec{r}(t)|^2 = R^2
$$
Или:
$$
r^2_0+t^2 + 2t(\vec{n}\cdot \vec{r}_0) = R^2
$$

Отсюда находим $t$:
$$
t = -(\vec{n}\cdot \vec{r}_0) \pm \sqrt{(\vec{n}\cdot \vec{r}_0)^2 - (r^2_0-R^2)}
$$

Выбираем корень с плюсом, потому что фотон рождён внутри сферы

In [16]:
def photon_propagation(pos: np.array, direction: np.array, R: float) -> np.array:
    nr = np.dot(pos, direction)
    r = np.linalg.norm(pos)

    t = -nr + (nr**2 - (r**2 - R**2))**0.5
    hit = pos + direction * t
    return hit

In [17]:
R = 2
N = 40
r = 0.1
a = 0.5

detector = Detector(R, N, r, a)
detector.fill_detector()
source_pos = np.array([0.5, 0.5, 0.5])
source = Source(source_pos)
photons = source.generate_photons(1.0)

In [18]:
charge = 0
for photon in photons:
    hit = photon_propagation(source_pos, photon, detector.R)
    charge += detector.register(hit)
    
expected_charge = detector.a * source.Y * (detector.N * detector.r**2/(4*detector.R**2))
print(f'expected charge : {int(expected_charge)}')
print(f'detected charge : {charge}')

expected charge : 125
detected charge : 140
